# VIIRS-GOES Candidate Overlap Viewer

Purpose: inspect whether GOES daily fire activity overlaps meaningful VIIRS candidate grids for the new component-wise spread formulation.

This notebook shows one `fire_id/date/component_id` example on the common `256x256` FirePred/VIIRS grid:

- VIIRS cumulative fire mask at day `d`
- VIIRS new growth at day `d+1`
- candidate grid points around the selected component
- GOES daily active/fire-frequency/FRP maps reprojected onto the same grid
- optional day `d+1` GOES panel as a leakage check only

For prediction `day d -> day d+1`, model features must use GOES observations up to `day d`, not `day d+1`.

In [ ]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage

REPO_ROOT = Path.cwd()
LEGACY_ROOT = REPO_ROOT / 'legacy'
if str(LEGACY_ROOT) not in sys.path:
    sys.path.insert(0, str(LEGACY_ROOT))

from scripts.analyze_pred_event_windows import load_daily_masks
from scripts.dataset_gen_pred_goes_spatial import (
    collect_goes_files_by_day,
    crop_profile,
    daily_maps,
    find_event_dir,
    firepred_path_from_viirs,
    parse_timestamp_from_name,
    viirs_day_files,
    FEATURE_NAMES,
)

CANDIDATE_ROOT = Path(os.environ.get('TS_SATFIRE_EVENT_CANDIDATE_ROOT', '/home/jlc3q/data/SatFire/event_candidates'))
GOES_ROOT = Path(os.environ.get('TS_SATFIRE_GOES_ROOT', '/home/jlc3q/data/GOES_clipped_tif_common_wgs84'))

SPLIT = os.environ.get('TS_SATFIRE_OVERLAP_SPLIT', 'test')
CANDIDATE_FILE = CANDIDATE_ROOT / f'pred_event_candidates_{SPLIT}_conn8_r5p0_mincomp1.csv'
SUMMARY_FILE = CANDIDATE_ROOT / f'pred_event_candidate_summary_{SPLIT}_conn8_r5p0_mincomp1.csv'

print('repo:', REPO_ROOT)
print('candidate file:', CANDIDATE_FILE)
print('GOES root:', GOES_ROOT)
print('split:', SPLIT)
assert CANDIDATE_FILE.exists(), CANDIDATE_FILE
assert SUMMARY_FILE.exists(), SUMMARY_FILE

## Select One Example

By default this picks a positive candidate from the test split. You can override the selected case by setting `FIRE_ID`, `DATE`, and `COMPONENT_ID` manually below.

In [ ]:
# Optional manual override. Leave as None to auto-select.
FIRE_ID = None
DATE = None
COMPONENT_ID = None

usecols = [
    'fire_id', 'date', 'next_date', 'day_idx', 'component_id',
    'candidate_row', 'candidate_col', 'nearest_fire_row', 'nearest_fire_col',
    'distance_px', 'direction_label_8', 'label_ignited_next_day',
]

# Read only positives first to avoid loading unnecessary columns.
df = pd.read_csv(CANDIDATE_FILE, usecols=usecols)
positives = df[df.label_ignited_next_day == 1].copy()
print('rows:', len(df), 'positives:', len(positives), 'positive_rate:', positives.shape[0] / len(df))

if FIRE_ID is None:
    # Prefer an example with many positive candidates in one component/day.
    key_counts = (
        positives.groupby(['fire_id', 'date', 'next_date', 'day_idx', 'component_id'])
        .size()
        .sort_values(ascending=False)
    )
    selected_key = key_counts.index[0]
    FIRE_ID, DATE, NEXT_DATE, DAY_IDX, COMPONENT_ID = selected_key
else:
    assert DATE is not None and COMPONENT_ID is not None
    subset0 = df[(df.fire_id == FIRE_ID) & (df.date == DATE) & (df.component_id == COMPONENT_ID)]
    assert len(subset0), 'manual selection produced no candidate rows'
    NEXT_DATE = subset0.next_date.iloc[0]
    DAY_IDX = int(subset0.day_idx.iloc[0])

case = df[(df.fire_id == FIRE_ID) & (df.date == DATE) & (df.component_id == COMPONENT_ID)].copy()
case_pos = case[case.label_ignited_next_day == 1]
print('selected:', FIRE_ID, DATE, '->', NEXT_DATE, 'day_idx=', DAY_IDX, 'component=', COMPONENT_ID)
print('case candidates:', len(case), 'case positives:', len(case_pos), 'positive_rate:', case_pos.shape[0] / len(case))
print(case_pos.direction_label_8.value_counts().sort_index())
case.head()

## Load VIIRS Masks and GOES Daily Maps

GOES daily maps are reprojected to the same `256x256` FirePred crop grid. For a `day d -> day d+1` prediction, the allowed GOES map is `day d`. The `day d+1` GOES map is displayed only to visually check leakage risk.

In [ ]:
# Test split has per-fire label_sel in ROI, but for this visual diagnostic label_sel=0 matches the original test pred task.
# Train/val use label_sel=1 by convention in the generator.
label_sel = 0 if SPLIT == 'test' else 1

dates, masks = load_daily_masks(FIRE_ID, label_sel)
print('n daily VIIRS masks:', len(masks))
print('date at DAY_IDX:', dates[DAY_IDX], 'next:', dates[DAY_IDX + 1])
assert dates[DAY_IDX] == DATE

current_mask = masks[DAY_IDX]
next_mask = masks[DAY_IDX + 1]
growth_mask = next_mask & ~current_mask

files = viirs_day_files(FIRE_ID)
ref_firepred = firepred_path_from_viirs(files[0])
dst_profile = crop_profile(ref_firepred)

event_dir = find_event_dir(GOES_ROOT, FIRE_ID)
print('GOES event dir:', event_dir)
assert event_dir is not None, f'No GOES event dir found for {FIRE_ID}'

goes_by_day = collect_goes_files_by_day(event_dir)
print('GOES days available:', len(goes_by_day))
print('GOES files on DATE:', {k: len(v) for k, v in goes_by_day.get(DATE, {}).items()})
print('GOES files on NEXT_DATE:', {k: len(v) for k, v in goes_by_day.get(NEXT_DATE, {}).items()})

goes_d = daily_maps(DATE, goes_by_day, dst_profile)
goes_next = daily_maps(NEXT_DATE, goes_by_day, dst_profile)

# Build candidate masks for the selected component.
candidate_mask = np.zeros_like(current_mask, dtype=bool)
positive_candidate_mask = np.zeros_like(current_mask, dtype=bool)
nearest_fire_mask = np.zeros_like(current_mask, dtype=bool)

candidate_mask[case.candidate_row.to_numpy(), case.candidate_col.to_numpy()] = True
if len(case_pos):
    positive_candidate_mask[case_pos.candidate_row.to_numpy(), case_pos.candidate_col.to_numpy()] = True
nearest_fire_mask[case.nearest_fire_row.to_numpy(), case.nearest_fire_col.to_numpy()] = True

print('current fire pixels:', int(current_mask.sum()))
print('growth pixels next day:', int(growth_mask.sum()))
print('case candidate pixels:', int(candidate_mask.sum()))
print('case positive candidate pixels:', int(positive_candidate_mask.sum()))
print('GOES DATE daily_active pixels:', int(goes_d['daily_active'].sum()))
print('GOES DATE active_frequency max:', float(np.nanmax(goes_d['active_frequency'])))
print('GOES DATE frp_sum_log1p max:', float(np.nanmax(goes_d['frp_sum_log1p'])))

## Overlay: VIIRS Current/Growth/Candidates

Red = current VIIRS fire state at day `d`. Blue = new growth at `d+1`. Yellow = candidate pixels for the selected component. Cyan = positive candidate pixels.

In [ ]:
def show_binary_overlay(ax, base=None, overlays=(), title=''):
    if base is None:
        base = np.zeros_like(current_mask, dtype=float)
    ax.imshow(base, cmap='gray', vmin=0, vmax=max(float(np.nanmax(base)), 1e-6))
    for mask, color, alpha, label in overlays:
        arr = np.where(mask, 1.0, np.nan)
        ax.imshow(arr, cmap=color, vmin=0, vmax=1, alpha=alpha)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])

fig, axes = plt.subplots(1, 3, figsize=(15, 5), dpi=120)
show_binary_overlay(
    axes[0],
    overlays=[
        (current_mask, 'Reds', 0.75, 'current'),
        (growth_mask, 'Blues', 0.75, 'growth'),
    ],
    title=f'VIIRS {DATE} -> {NEXT_DATE}',
)
show_binary_overlay(
    axes[1],
    overlays=[
        (current_mask, 'Reds', 0.45, 'current'),
        (candidate_mask, 'Wistia', 0.55, 'candidates'),
        (positive_candidate_mask, 'winter', 0.95, 'positive candidates'),
    ],
    title=f'component {COMPONENT_ID}: candidates',
)
show_binary_overlay(
    axes[2],
    overlays=[
        (nearest_fire_mask, 'Reds', 0.8, 'nearest front'),
        (positive_candidate_mask, 'winter', 0.95, 'positive candidates'),
    ],
    title='nearest front pixels -> positives',
)
plt.tight_layout()

## Overlay: GOES Day-d Daily Activity on VIIRS Grid

These panels use only GOES observations from `DATE`, which is allowed for a `DATE -> NEXT_DATE` prediction.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10), dpi=120)

panels = [
    ('GOES daily_active', goes_d['daily_active'], 'Reds'),
    ('GOES active_frequency', goes_d['active_frequency'], 'magma'),
    ('GOES frp_sum_log1p', goes_d['frp_sum_log1p'], 'inferno'),
    ('GOES frp_max_log1p', goes_d['frp_max_log1p'], 'inferno'),
    ('VIIRS growth + GOES active', goes_d['daily_active'], 'Reds'),
    ('Candidates + GOES FRP', goes_d['frp_sum_log1p'], 'inferno'),
]

for ax, (title, img, cmap) in zip(axes.ravel(), panels):
    vmax = np.nanpercentile(img[img > 0], 99) if np.any(img > 0) else 1
    ax.imshow(img, cmap=cmap, vmin=0, vmax=max(float(vmax), 1e-6))
    ax.imshow(np.where(current_mask, 1, np.nan), cmap='Greys', alpha=0.35, vmin=0, vmax=1)
    if 'growth' in title:
        ax.imshow(np.where(growth_mask, 1, np.nan), cmap='Blues', alpha=0.85, vmin=0, vmax=1)
    if 'Candidates' in title:
        ax.imshow(np.where(candidate_mask, 1, np.nan), cmap='Wistia', alpha=0.35, vmin=0, vmax=1)
        ax.imshow(np.where(positive_candidate_mask, 1, np.nan), cmap='winter', alpha=0.95, vmin=0, vmax=1)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle(f'{FIRE_ID} | allowed GOES date={DATE}', y=1.02)
plt.tight_layout()

## Candidate-Level GOES Values

This samples the reprojected GOES day-d raster at candidate row/col. This is the direct overlap feature we can join into the candidate table.

In [ ]:
case_eval = case.copy()
rr = case_eval.candidate_row.to_numpy()
cc = case_eval.candidate_col.to_numpy()
for name, arr in goes_d.items():
    case_eval[f'goes_{name}_at_candidate'] = arr[rr, cc]

cols = [
    'label_ignited_next_day', 'distance_px', 'direction_label_8',
    'goes_daily_active_at_candidate',
    'goes_active_frequency_at_candidate',
    'goes_frp_sum_log1p_at_candidate',
    'goes_frp_max_log1p_at_candidate',
]
print(case_eval[cols].groupby('label_ignited_next_day').agg(['mean', 'max', 'count']))
case_eval[cols].sort_values('goes_frp_sum_log1p_at_candidate', ascending=False).head(20)

## Leakage Check: GOES Day d+1

The next panel shows GOES from `NEXT_DATE`. This should **not** be used as a feature for predicting `NEXT_DATE` growth. It is shown only to confirm how much future GOES would trivially reveal target fire activity.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), dpi=120)
for ax, (title, img, cmap) in zip(
    axes,
    [
        (f'allowed GOES active {DATE}', goes_d['daily_active'], 'Reds'),
        (f'FORBIDDEN GOES active {NEXT_DATE}', goes_next['daily_active'], 'Reds'),
        (f'VIIRS growth {NEXT_DATE}', growth_mask.astype(float), 'Blues'),
    ],
):
    ax.imshow(img, cmap=cmap, vmin=0, vmax=max(float(np.nanmax(img)), 1e-6))
    ax.imshow(np.where(current_mask, 1, np.nan), cmap='Greys', alpha=0.35, vmin=0, vmax=1)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()

## What To Look For

- If allowed GOES day-d activity overlaps positive candidates more than negative candidates, direct GOES overlap is useful.
- If only forbidden day-d+1 GOES overlaps growth, then using future GOES would cause leakage and should be avoided.
- If GOES activity is coarse but roughly near the selected component, use local-window features such as 3x3/5x5 max or distance-to-nearest-GOES-active instead of only point sampling.